In [ ]:
!pip install sentence-transformers

## Import Libraries

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

## Load and Combine News Datasets

In [7]:
folder = Path("/Users/sharon/Downloads/formatted_news")

dfs = []

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    dfs.append(df)

all_news = pd.concat(dfs, ignore_index=True)

all_news.head()

,date,title,description,url
0,2019-01-04,What the Constitution Means to Me review – a f...,"Heidi Schreck’s funny, impactful and breathles...",https://www.theguardian.com/stage/2019/mar/31/...
1,2019-01-04,Kim Jong-nam poisoning trial: last suspect to ...,Doan Thi Huong pleaded guilty to ‘causing hurt...,https://www.theguardian.com/world/2019/apr/01/...
2,2019-01-04,Inquiry into Paladin expanded to cover all Man...,Auditor general to assess whether offshore pro...,https://www.theguardian.com/australia-news/201...
3,2019-01-04,Reiwa: Japan prepares to enter new era of 'for...,"New era will go into effect on 1 May, a day af...",https://www.theguardian.com/world/2019/apr/01/...
4,2019-01-04,Lord Howe Island coral bleaching 'most severe ...,Biologists fear they will now start to see cor...,https://www.theguardian.com/australia-news/201...


In [8]:
print(all_news.shape)

(2934251, 4)


## Data Cleaning & Text Preprocessing

In [9]:
# Remove Mssing value & Duplicate
all_news = all_news.dropna(subset=["date", "title", "description", "url"])
all_news = all_news.drop_duplicates(subset=[ "url"])

all_news.reset_index(drop=True, inplace=True)

print(all_news.shape)

(2260174, 4)


In [14]:
# Cleaned description
all_news["short_description"] = (
    all_news["description"]
    .fillna("")
    .str.replace(r"\(Category:.*?\)\.?", "", regex=True)
    .str.replace(r"Media coverage was largely \w+\.", "", regex=True)
    .str.replace(r"This event was covered by \d+ articles\.", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Construct Embedding Text
all_news["text_for_embedding"] = (
    all_news["title"].fillna("") + ". " +
    all_news["short_description"].str[:500]
)

In [16]:
print(all_news.shape)
all_news["text_for_embedding"].head()


(2260174, 6)


0    What the Constitution Means to Me review – a f...
1    Kim Jong-nam poisoning trial: last suspect to ...
2    Inquiry into Paladin expanded to cover all Man...
3    Reiwa: Japan prepares to enter new era of 'for...
4    Lord Howe Island coral bleaching 'most severe ...
Name: text_for_embedding, dtype: object

## Generating Embedding

In [11]:
# Load Embedding Model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
embeddings = model.encode(
    all_news["text_for_embedding"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embeddings.shape)

Batches:   0%|          | 0/35316 [00:00<?, ?it/s]

(2260174, 384)


## Save

In [18]:
import numpy as np
from pathlib import Path

# Save Processed Data and Embeddings
output_folder = Path("/Users/sharon/Downloads/embedding_output")
output_folder.mkdir(parents=True, exist_ok=True)

all_news.to_csv(output_folder / "all_news.csv", index=False)
np.save(output_folder / "news_embeddings.npy", embeddings)

## Load csv & embedding

In [ ]:
all_news = pd.read_csv("/Users/sharon/Downloads/embedding_output/all_news.csv")
embeddings = np.load("/Users/sharonDownloads/embedding_output/news_embeddings.npy")

## Retrieval Function

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
def get_articles_from_query(user_query, top_k=5):

    # BGE recommend instruction
    query = (
        "Represent this sentence for searching relevant passages: "
        + user_query
    )

    # query embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # cosine similarity
    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # find relevent news
    top_indices = np.argsort(scores)[::-1][:top_k]

    # output
    articles = []

    for idx in top_indices:

        articles.append({
            "date": all_news.loc[idx, "date"],
            "title": all_news.loc[idx, "title"],
            "description": all_news.loc[idx, "description"],
            "url": all_news.loc[idx, "url"],
            #"score": float(scores[idx])
        })

    return articles

In [23]:
results = get_articles_from_query(
    "AI regulations in 2023",
    top_k=5
)

results

[{'date': '2025-07-06',
  'title': 'UK ministers delay AI regulation amid plans for more ‘comprehensive’ bill',
  'description': 'Law expected to include safety and copyright issues but delay likely to raise concerns about ongoing lack of regulation',
  'url': 'https://www.theguardian.com/technology/2025/jun/07/uk-ministers-delay-ai-regulation-amid-plans-for-more-comprehensive-bill'},
 {'date': '2023-02-06',
  'title': 'Australia is looking to regulate AI – what might they be used for and what could go wrong?',
  'description': 'Growing sense that artificial intelligence is in accelerated development prompts government review looking to make ‘modern laws for modern technology’',
  'url': 'https://www.theguardian.com/technology/2023/jun/03/australia-is-looking-to-regulate-ai-what-might-they-be-used-for-and-what-could-go-wrong'},
 {'date': '2023-05-24',
  'title': 'business artificial intelligence regulation openai',
  'description': 'CONGRESS appealed to INTELLIGENCE in Connecticut, Uni

In [25]:
results = get_articles_from_query(
    "covid",
    top_k=15
)

results

[{'date': '2020-03-05',
  'title': 'Covid-19 has put cancer patients and research at huge risk',
  'description': '<strong>Letters: </strong>Patients are missing out on vital care and charities funding research have been hit by the pandemic, says <strong>Prof</strong><strong> Ara Darzi </strong>',
  'url': 'https://www.theguardian.com/world/2020/may/03/covid-19-has-put-cancer-patients-and-research-at-huge-risk'},
 {'date': '2020-06-05',
  'title': "'People are dying at home': virus fears deter seriously ill from hospitals",
  'description': 'Covid-19 has swamped healthcare across the US but doctors have noticed a drop in admissions for common ailments such as heart attacks and strokes',
  'url': 'https://www.theguardian.com/world/2020/may/06/dying-at-home-non-covid-19-hospitals-coronavirus'},
 {'date': '2020-11-07',
  'title': 'Covid-19 has revealed a pre-existing pandemic of poverty that benefits the rich',
  'description': 'Governments must take seriously the human right to an adequa

In [29]:
# Diversifies results across years
def get_articles_from_query(user_query, top_k=10):

    # query embedding
    query = (
        "Represent this sentence for searching relevant passages: "
        + user_query
    )

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # similarity
    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    temp_df = all_news.copy()
    temp_df["score"] = scores

    # get year
    temp_df["year"] = pd.to_datetime(
        temp_df["date"]
    ).dt.year

    # top 200
    temp_df = temp_df.sort_values(
        "score",
        ascending=False
    ).head(200)

    # 2 each year
    results = []

    for year in sorted(temp_df["year"].unique()):

        yearly_news = (
            temp_df[temp_df["year"] == year]
            .sort_values("score", ascending=False)
            .head(2)
        )

        results.append(yearly_news)

    # combine
    results_df = pd.concat(results)

    # sort base on date
    results_df = results_df.sort_values("date")

    articles = []

    for _, row in results_df.iterrows():

        articles.append({
            "date": row["date"],
            "title": row["title"],
            "description": row["description"],
            "url": row["url"],
            #"score": float(row["score"])
        })

    return articles[:top_k]

In [30]:
results = get_articles_from_query(
    "covid",
    top_k=15
)

results

[{'date': '2020-03-05',
  'title': 'Covid-19 has put cancer patients and research at huge risk',
  'description': '<strong>Letters: </strong>Patients are missing out on vital care and charities funding research have been hit by the pandemic, says <strong>Prof</strong><strong> Ara Darzi </strong>',
  'url': 'https://www.theguardian.com/world/2020/may/03/covid-19-has-put-cancer-patients-and-research-at-huge-risk'},
 {'date': '2020-06-05',
  'title': "'People are dying at home': virus fears deter seriously ill from hospitals",
  'description': 'Covid-19 has swamped healthcare across the US but doctors have noticed a drop in admissions for common ailments such as heart attacks and strokes',
  'url': 'https://www.theguardian.com/world/2020/may/06/dying-at-home-non-covid-19-hospitals-coronavirus'},
 {'date': '2021-10-01',
  'title': 'Covid has undermined chronically under-funded justice system',
  'description': 'Analysis: coronavirus has exacerbated the crisis facing courts but political co